# 第 1 周末练习 —— 有状态技术问答（Chat 类）

## 练习目标（理念）

在 **OpenAI Chat Completions API** 之上做一个技术问答工具，并额外练会「多轮对话」。

**本笔记本演示：**

- **有状态对话（Stateful Chat）**：`Chat` 类维护 `history`，每轮把完整上下文发给模型
- **流式响应（Streaming）**：`ask_stream` 边收边 `print`
- **多模型对比**：同一套 `Chat`，换客户端即可问 GPT / Llama / Gemini
- **对话实用工具**：查看历史、`reset` 清空后重新开始

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create(...)` |
| system / user / assistant | `history` 里按 role 追加 |
| `stream=True` | `ask_stream` 逐 token 打印 |
| OpenAI 兼容端点 | Ollama、OpenRouter 只换 `base_url` + `api_key` |

## 怎么跑

1. 准备 `.env` 中的 `OPENAI_API_KEY`；Llama 需本地 Ollama；Gemini 需 `OPENROUTER_API_KEY`
2. 从上到下运行；在提问格改 `question` 后重跑对应模型格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（例如 OPENAI_API_KEY、OPENROUTER_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown 与 display：在笔记本里漂亮渲染回答
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：云端官方、Ollama、OpenRouter 都可用同一类
from openai import OpenAI


In [ ]:
# ========== 常量：默认模型名集中管理 ==========

# 云端小模型：便宜、适合解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 pull，且与本机名称一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境：加载密钥并创建默认 OpenAI 客户端 ==========

# override=True：.env 中的值覆盖已有同名环境变量
load_dotenv(override=True)
# 读取官方 OpenAI 密钥
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：是否像 sk-proj- 开头且足够长（报错/提示文案保持英文）
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 使用默认环境配置创建客户端（会自动读 OPENAI_API_KEY）
openai = OpenAI()


## 助手工具函数

下面先定义一个小助手，把普通字符串渲染成笔记本里的 Markdown。


In [ ]:
# ========== 助手：把文本显示为 Markdown ==========

def display_md(text):
    """Render text as Markdown in the notebook."""
    # display(Markdown(...))：在 Jupyter 输出区渲染格式化文本
    display(Markdown(text))


## 聊天类 —— 有状态的对话包装器

OpenAI Chat Completions API **本身无状态**：单次请求看不到「上一轮说了什么」。

本练习的 `Chat` 类用列表 `history` 模拟多轮对话：

- 每次用户提问 → 追加一条 `user`
- 每次模型回答 → 追加一条 `assistant`
- 真正调用 API 时：`[system] + history` 整包发送，模型才能引用前文

这就像给无记忆的接口外挂了一个「记事本」。


In [ ]:
# ========== Chat 类：在无状态 API 上模拟多轮对话 ==========

class Chat:
    """Stateful chat wrapper around OpenAI's completions API."""

    def __init__(self, client, system="You are a helpful technical assistant.", model=MODEL_GPT):
        # 注入的 OpenAI 兼容客户端（官方 / Ollama / OpenRouter 皆可）
        self.client = client
        # 本会话默认模型 id
        self.model = model
        # system prompt 字符串（发给模型的指令，保持英文默认值）
        self.system = system
        # history：只存 user + assistant；system 每次临时拼在最前
        self.history = []  # stores user + assistant messages

    def ask(self, question, show=True):
        """Send a question and get a complete response."""
        # 先把用户问题记入历史
        self.history.append({"role": "user", "content": question})
        # 每次请求都带上 system + 完整 history
        messages = [{"role": "system", "content": self.system}] + self.history
        # 非流式：等整段生成完再取 content
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages
        )
        # 从 choices[0].message 取出助手全文
        reply = response.choices[0].message.content
        # 把助手回复也写入历史，供下一轮引用
        self.history.append({"role": "assistant", "content": reply})
        # show=True 时在笔记本里渲染 Markdown
        if show:
            display_md(reply)
        return reply

    def ask_stream(self, question):
        """Send a question and stream the response token-by-token."""
        # 与 ask 一样：先记录 user
        self.history.append({"role": "user", "content": question})
        messages = [{"role": "system", "content": self.system}] + self.history
        # stream=True：返回可迭代的增量事件
        stream = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            stream=True
        )
        # chunks：收集每个 delta 文本，最后拼成完整 reply 写入 history
        chunks = []
        for chunk in stream:
            # 增量在 choices[0].delta.content
            delta = chunk.choices[0].delta.content
            if delta:
                # end='' 不换行；flush=True 立刻刷到屏幕 → 打字机效果
                print(delta, end='', flush=True)
                chunks.append(delta)
        # 流结束后补一个换行，避免下一行输出粘在一起
        print()
        # 拼出完整回复并记入历史
        reply = ''.join(chunks)
        self.history.append({"role": "assistant", "content": reply})
        # 再用 Markdown 漂亮显示一遍完整回答
        display_md(reply)

    def show_history(self):
        """Display the full conversation history."""
        # 先展示 system，再按 role 逐条展示 history
        display_md(f'**[SYSTEM]**\n\n{self.system}\n\n---')
        for msg in self.history:
            role = msg["role"].upper()
            display_md(f'**[{role}]**\n\n{msg["content"]}\n\n---')

    def reset(self):
        """Clear conversation history to start fresh."""
        # 清空 user/assistant 历史；system 仍保留在 self.system
        self.history = []


## 询问 GPT-4o-mini（流式）

用官方 OpenAI 客户端 + `MODEL_GPT`，走 `ask_stream` 看打字机效果。


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的内容保持英文；可换成你自己困惑的代码片段
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 路径 A：GPT-4o-mini 流式问答 ==========

# 用前面创建的官方 openai 客户端，模型取常量 MODEL_GPT
gpt_chat = Chat(openai, model=MODEL_GPT)
# ask_stream：边生成边打印，并把完整回复写入 gpt_chat.history
gpt_chat.ask_stream(question)


## 询问 Llama 3.2（通过 Ollama）

Ollama 在本地暴露了 **OpenAI 兼容** 的 `/v1` API，因此可以**复用同一个 `Chat` 类**：

- 只换 `OpenAI(base_url=..., api_key=...)`
- 只换 `model=MODEL_LLAMA`
- 对话逻辑（history / ask / ask_stream）一行都不用改

前提：本机 Ollama 在跑，且已 `ollama pull llama3.2`。


In [ ]:
# ========== 路径 B：本地 Llama（Ollama OpenAI 兼容端点） ==========

# base_url 指向本机 11434；api_key 占位字符串即可（Ollama 会忽略）
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# 新的 Chat 实例：独立 history，不会和 gpt_chat 串台
llama_chat = Chat(ollama_client, model=MODEL_LLAMA)
# 非流式 ask：等整段完成再 Markdown 显示
llama_chat.ask(question)


## 查看历史记录

检查 `gpt_chat` 里已经积累的 system / user / assistant，确认「有状态」是否生效。


In [ ]:
# ========== 打印 gpt_chat 的完整对话历史 ==========
gpt_chat.show_history()


## 重置

清空 `gpt_chat.history`，方便用同一实例重新开聊（system 设定仍保留）。


In [ ]:
# ========== 清空 gpt_chat 的 user/assistant 历史 ==========
gpt_chat.reset()


## 询问 Gemini（通过 OpenRouter）

OpenRouter 同样提供 OpenAI 兼容端点：继续复用 `Chat`，只需：

1. 从环境变量读取 `OPENROUTER_API_KEY`
2. `base_url` 换成 OpenRouter
3. `model` 换成 Gemini 的路由字符串


In [ ]:
# ========== 路径 C：经 OpenRouter 创建 Gemini 客户端 ==========

# 读取 OpenRouter 密钥（不要写进笔记本正文）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
# Gemini 在 OpenRouter 上的模型 id（字符串保持原样）
GEMINI_MODEL = 'google/gemini-3-flash-preview'

# 指向 OpenRouter 的 OpenAI 兼容客户端
gemini_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1/"
)
# 初始化成功提示（文案保持英文）
print("Gemini client initialized successfully.")

# 用同一 Chat 类包装；之后可 gemini_chat.ask(...)
gemini_chat = Chat(gemini_client, model=GEMINI_MODEL)


In [ ]:
# ========== 用 Gemini Chat 实例回答同一个 question ==========
gemini_chat.ask(question)
